# AgriConnect: Crop Sales Optimization & Price Prediction Model
**Course**: DIDT3110 - Digital Strategy for Business Transformation  
**Faculty**: Information & Communication Technology - Limkokwing University Lesotho  
**Part D**: Artificial Intelligence (50 Marks)  
**Objective**: Leverage disruptive AI technology to optimize crop sales pricing, forecast agricultural market trends, and understand algorithm accuracy & saturation levels.

## 1. Environment Setup & Data Loading

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load dataset
df = pd.read_csv('../data/lesotho_crop_market_dataset.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (1200, 22)


,record_id,date,season,district,crop_name,category,farm_size_ha,rainfall_mm,temperature_c,actual_price_per_kg_lsl
0,AGC-00001,2023-05-21,Autumn,Butha-Buthe,Beans (Dry),legumes,6.83,62.6,9.8,21.86
1,AGC-00002,2024-05-27,Autumn,Leribe,Wheat,grains,3.95,40.5,16.7,12.97
2,AGC-00003,2026-01-12,Summer,Butha-Buthe,Potatoes,vegetables,0.91,163.6,21.8,9.02
3,AGC-00004,2025-09-25,Spring,Leribe,Peaches,fruits,9.09,90.6,19.8,24.14
4,AGC-00005,2024-07-18,Winter,Qacha's Nek,Tomatoes,vegetables,9.17,5.0,4.7,16.65


## 2. Exploratory Data Analysis (EDA)
Examining price distributions across Lesotho districts and crop varieties.

In [2]:
df.describe()

# Distribution of price by crop
plt.figure(figsize=(12, 5))
sns.boxplot(x='crop_name', y='actual_price_per_kg_lsl', data=df, palette='Set2')
plt.title('Crop Price Distribution in Lesotho (Maloti/kg)')
plt.xlabel('Crop Name')
plt.ylabel('Price (LSL/kg)')
plt.xticks(rotation=30)
plt.grid(alpha=0.3)
plt.show()

,farm_size_ha,rainfall_mm,temperature_c,soil_ph,fertilizer_kg_per_ha,historical_avg_price_lsl,competitor_supply_tons,market_demand_index,transport_cost_lsl,actual_price_per_kg_lsl
count,1200.000,1200.000,1200.000,1200.000,1200.000,1200.000,1200.000,1200.000,1200.000,1200.000
mean,6.012,75.843,14.628,6.471,184.731,13.247,129.584,50.142,248.316,14.892
std,2.892,44.217,5.934,0.544,93.158,5.831,69.447,28.853,116.248,6.734
min,0.500,5.000,2.100,5.400,20.000,5.200,10.000,5.000,50.000,5.120
25%,3.590,38.400,9.700,5.980,105.000,8.300,73.000,24.000,145.000,9.370
50%,5.955,71.550,14.650,6.470,186.000,12.500,129.500,50.000,249.000,13.845
75%,8.370,112.325,19.600,6.970,265.000,18.100,184.000,76.000,351.000,20.148
max,12.000,174.000,25.800,7.500,350.000,24.800,250.000,99.000,450.000,29.950


## 3. Feature Engineering & Preprocessing Pipeline

In [3]:
categorical_features = ['district', 'crop_name', 'season', 'category']
numeric_features = [
    'farm_size_ha', 'rainfall_mm', 'temperature_c', 'soil_ph',
    'fertilizer_kg_per_ha', 'historical_avg_price_lsl',
    'competitor_supply_tons', 'market_demand_index', 'transport_cost_lsl'
]

X = df[categorical_features + numeric_features]
y = df['actual_price_per_kg_lsl']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape[0]} samples | Test set: {X_test.shape[0]} samples")

Training set: 960 samples | Test set: 240 samples


## 4. Model Training & Benchmarking
We evaluate 4 distinct algorithms:
1. **Random Forest Regressor** (Ensemble Bagging)
2. **Gradient Boosting Regressor** (Ensemble Boosting)
3. **Decision Tree Regressor** (Single CART Tree)
4. **Multiple Linear Regression** (Parametric Baseline)

In [4]:
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    'Decision Tree': DecisionTreeRegressor(max_depth=8, random_state=42),
    'Linear Regression': LinearRegression()
}

metrics = []
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    
    metrics.append({
        'Model': name,
        'R² Score': round(r2, 4),
        'RMSE (LSL)': round(rmse, 3),
        'MAE (LSL)': round(mae, 3)
    })

metrics_df = pd.DataFrame(metrics)
metrics_df

,Model,R² Score,RMSE (LSL),MAE (LSL)
0,Random Forest,0.9242,1.148,0.843
1,Gradient Boosting,0.9085,1.278,0.962
2,Decision Tree,0.8315,1.764,1.321
3,Linear Regression,0.7621,2.103,1.648


## 5. Algorithm Saturation Analysis (Part D.3)
### A. Sample Size Saturation (Learning Curve)
This illustrates the point where additional training records provide diminishing returns.

In [5]:
best_pipe = Pipeline([('preprocessor', preprocessor), ('model', RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42))])
train_sizes, train_scores, test_scores = learning_curve(
    best_pipe, X, y, cv=5, scoring='r2',
    train_sizes=np.linspace(0.1, 1.0, 10), random_state=42
)

plt.figure(figsize=(10, 5))
plt.plot(train_sizes, np.mean(train_scores, axis=1), 'o-', color='green', label='Training R² Score')
plt.plot(train_sizes, np.mean(test_scores, axis=1), 'o-', color='blue', label='Cross-Validation R² Score')
plt.axvline(x=850, color='red', linestyle='--', label='Data Saturation Threshold (~850 samples)')
plt.title('Learning Curve & Data Saturation Plateau')
plt.xlabel('Training Samples (N)')
plt.ylabel('R² Score')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### B. Hyperparameter Depth Saturation (Bias-Variance Trade-off)
Visualizing the optimal maximum depth before overfitting occurs.

In [6]:
depths = range(2, 26, 2)
train_acc = []
test_acc = []

for d in depths:
    pipe = Pipeline([('preprocessor', preprocessor), ('model', RandomForestRegressor(n_estimators=50, max_depth=d, random_state=42))])
    pipe.fit(X_train, y_train)
    train_acc.append(r2_score(y_train, pipe.predict(X_train)))
    test_acc.append(r2_score(y_test, pipe.predict(X_test)))

plt.figure(figsize=(10, 5))
plt.plot(depths, train_acc, 'o-', color='purple', label='Train R² (Memorization)')
plt.plot(depths, test_acc, 's-', color='orange', label='Test R² (Generalization)')
plt.axvspan(8, 12, color='green', alpha=0.15, label='Optimal Saturation Zone (Depth 8-12)')
plt.title('Hyperparameter Saturation: Tree Depth vs. Model Performance')
plt.xlabel('Max Depth')
plt.ylabel('R² Score')
plt.legend()
plt.grid(alpha=0.3)
plt.show()